In [1]:
import pandas as pd
import unicodedata
import re
import numpy as np

from mappings import *
from col_renames import *
from helper_functions import *

In [2]:
df = pd.read_csv("../data.csv", encoding="utf-8")

In [3]:
df.shape

(233, 83)

Removing Bogus or Nonsensical data before NaN removal

In [4]:
# 11 : residence_country = Bides

drop_index = [11]
df.drop(index=drop_index, inplace=True)

In [5]:
nan_counts = df.isna().sum()
print(nan_counts[nan_counts > 0])

Which industry are you working for?                                                  1
What is your occupation?                                                             1
Having national laws that enforce ethical and socially responsible AI use            1
Voluntary certifications from trusted organizations ensuring ethical standards       1
Independent experts monitoring AI use and informing the public                       2
Companies following a self-regulated code of ethics for AI development               2
Clear and transparent explanations about what the AI does and how it uses data       2
Inclusion of diverse teams and stakeholder consultation throughout AI development    3
dtype: int64


Since the number of NaN values is few and we are only working with a subset of the data, we will be dropping any rows with nan values

In [6]:
df = df.dropna()

In [7]:
if df.isna().any().any():
    print("Yes, NaN values exist")
else:
    print("No, NaN values do not exist")

No, NaN values do not exist


In [8]:
df.shape

(225, 83)

In [9]:
# Cleaning all the values to be lower, stripped and ascii only, could break hash_id column
df[df.columns] = df[df.columns].map(to_ascii)

In [10]:
df = rename_cols(df, cols_to_positional_extract)
df = rename_cols(df, cols_to_parse)
df = rename_cols(df, col_one_hot)
df = rename_cols(df, col_binary)
df = rename_cols(df, col_ordinal)
df = rename_cols(df, col_ignore)

In [11]:
# getting the lists for the appropriate type of category easier further processing
positional_extract_cols = list(cols_to_positional_extract.values())
parse_cols = list(cols_to_parse.values())
one_hot_cols = list(col_one_hot.values())
binary_cols = list(col_binary.values())
ordinal_cols = list(col_ordinal.values())
ignored_cols = list(col_ignore.values())

In [12]:
# Manually executing all the mappings
df["ethnicity"] = df["ethnicity"].replace(ethnicity_map)
df["residence_country"] = df["residence_country"].replace(region_map)
df["residence_type"] = df["residence_type"].replace(residence_type_map)
df['highest_education'] = df['highest_education'].replace(education_map)

In [13]:
# Converting all the values that can be preprocessed into binary values
for col in binary_cols:
    df[col] = (
        df[col]
        .str.strip()
        .str.lower()
        .eq("yes"))

df[binary_cols] = df[binary_cols].astype(np.uint16)

In [14]:
df = df.rename(columns = {"What language(s) do you speak? If you speak multiple, please provide in comma separated format.\nExample, Nepali, Tharu, Turkish and Hindi": "language"})

In [15]:
language_series = df["language"]

language_series = language_series.apply(lambda x : x.lower().replace(",", ";").replace(".", ";"))
language_series = language_series.apply(lambda x: re.sub(r"/", ";", x))
language_series = language_series.apply(lambda x: re.sub(r" +", ";", x))

df["language"] = language_series

In [16]:
df["language"] = df["language"].apply(map_and_categorize, maps=language_map, default_to_self=True)

language = df["language"].str.get_dummies(sep=";").astype(np.uint16)
language = language.add_prefix("language_")
df = df.join(language)

In [17]:
# Converting all the values that can be preprocessed with one_hot
df = pd.get_dummies(
    df,
    columns=one_hot_cols,
    dtype=np.uint16,
    drop_first=True
)

In [18]:
for col in ordinal_cols:
    df[col] = df[col].map(ordinal_maps[col]).astype(np.uint16)

In [19]:
# Renaming the two as they were omitted in the rename mappings
df.rename(columns = {"Which industry are you working for?": "industry"}, inplace=True)
df.rename(columns = {"What is your occupation?": "occupation"}, inplace=True)

In [20]:
df["occupation"] = df["occupation"].replace(occupation_map)
df["industry"] = df["industry"].replace(industry_map)

In [21]:
occupation = pd.get_dummies(df["occupation"], dtype=np.uint16)
industry = pd.get_dummies(df["industry"], dtype=np.uint16)

occupation = occupation.add_prefix("occupation_")
industry = industry.add_prefix("industry_")

df = df.join(occupation)
df = df.join(industry)

In [22]:
df.drop(columns=["language", "industry", "occupation", "ranking_instruction_only"], inplace=True)

In [23]:
build_ai_factors = [
    "protecting privacy and keeping personal data secure",
    "having humans review and control ai decisions",
    "treating everyone fairly and making ai accessible to all",
    "managing risks and being clear about who is responsible",
    "clearly explaining what the ai does and its limitations",
    "considering how ai affects society and the environment",
    "ensuring accurate results and protecting data from misuse"
]

ethical_factors = [
    "data privacy",
    "transparency",
    "accessibility",
    "affordability",
    "non-discrimination",
    "accountability"
]

In [24]:
def score_encode(row, factors):
    """To encode the appropriate scores"""
    items = row.split(";")
    return {factor : len(factors) - items.index(factor) for factor in factors}

In [25]:
# Calculating all the different scores that are based on position
build_scores = (
    df["rank_most_important_factors_when_building_ai"]
        .apply(score_encode, factors=build_ai_factors)
        .apply(pd.Series)
        ).add_prefix("rank_most_important_factors_when_building_ai_").astype(np.uint16)

ethics_scores = (
    df["rank_ethical_priorities_in_country"].apply(erase_parentheses)
        .apply(score_encode, factors=ethical_factors)
        .apply(pd.Series)
        ).add_prefix("rank_ethical_priorities_in_country_").astype(np.uint16)

df = df.join([build_scores, ethics_scores])
df.drop(columns=[
    "rank_most_important_factors_when_building_ai",
    "rank_ethical_priorities_in_country"], inplace=True)

In [26]:
df["benefit_of_ai_to_country"] = (
    df["benefit_of_ai_to_country"]
    .apply(map_and_categorize, maps=benefit_to_country_map, default_to_self=False))

df["concerns_about_ai_use_in_country"] = (
    df["concerns_about_ai_use_in_country"]
    .apply(map_and_categorize, maps=concerns_about_ai_map, default_to_self=False))

In [27]:
# Cleaning the up
df["ai_powered_tools_used"] = df["ai_powered_tools_used"].apply(erase_parentheses)
df["applications_thought_to_use_ai"] = df["applications_thought_to_use_ai"].apply(erase_parentheses)

# Applying proper categorization and mapping
df["ai_powered_tools_used"] = df["ai_powered_tools_used"].apply(map_and_categorize, maps=ai_tools_map, default_to_self=True)
df["applications_thought_to_use_ai"] = df["applications_thought_to_use_ai"].apply(map_and_categorize, maps=app_use_ai_map, default_to_self=True)

In [28]:
parsed_df = pd.DataFrame(index=df.index)

for col in parse_cols:
    temp = df[col].str.get_dummies(sep=";").astype(np.uint16)
    temp = temp.add_prefix(f"{col}_")
    parsed_df = parsed_df.join(temp)

df = df.join(parsed_df)

In [29]:
# Dropping all the cols of parse_cols as we manually removed all of them
df.drop(columns=parse_cols, inplace=True)

In [30]:
# Final Sanity Check
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 225 entries, 0 to 232
Columns: 137 entries, row_id to applications_thought_to_use_ai_travel/mobility
dtypes: object(5), uint16(132)
memory usage: 68.6+ KB


In [31]:
# Outputting the file
df.to_csv("../output.csv")